# Runs all QM8 experiments with the full dataset!

16 output tasks 

QM8 Regression, 21,000 ish compounds, Random split, MAE

Multitask: 0.0150  graph: MPNN: 0.0143

Use tdaf-tf2p7h2             C:\Users\ella_\.conda\envs\tdaf-tf2p7h2

In [1]:
import sys
sys.path.append(r"C:\Users\ella_\Documents\GitHub\graphs_and_topology_for_chemistry")
sys.path.append(r"C:\Users\ella_\Documents\GitHub\icosahedron_projection")

import deepchem as dc
#from deepchem.feat import SmilesToImage

import tensorflow as tf
import os
import sys
import rdkit
import h5py
import helper_functions as h

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.tri
import rdkit.Chem as Chem
import rdkit.Chem.AllChem as AllChem
from rdkit.Chem import Descriptors
from rdkit.Chem import rdMolDescriptors
import mpl_toolkits.mplot3d
from sklearn.ensemble import RandomForestRegressor
from sklearn.decomposition import PCA
from collections import Counter

print("TensorFlow version: " + tf.__version__)

# topology stuff
from gtda.plotting import plot_point_cloud
from gtda.homology import VietorisRipsPersistence
from gtda.plotting import plot_diagram
from gtda.diagrams import PersistenceEntropy
from gtda.diagrams import NumberOfPoints
from gtda.diagrams import Amplitude
from sklearn.pipeline import make_union, Pipeline

# fixc this at some point
sys.path.append(r"C:\Users\ella_\Documents\GitHub\graphs_and_topology_for_chemistry")
sys.path.append(r"C:\Users\ella_\Documents\GitHub\icosahedron_projection")

import projection
#from projection.molecule import Molecule
#from projection.pdbmolecule import PDBMolecule
#from projection.mol2molecule import Mol2Molecule

import helper_functions as h
loaders, classification_datasets, regression_datasets, metric_types = h.deepchem_dataset_dictionaries()

# change THIS

dataset_name='qm8'

data_dir=r'F:\Nextcloud\science\Datasets\topol_datasets'
results_dir=r"F:\Nextcloud\science\results\topology_and_graphs\d_" + dataset_name
test_file=dataset_name + '.csv'
data_file_name=dataset_name + '_topological_features.hdf5'
loader = loaders[dataset_name]
make_dataset=False # whether to recalc the dataset

print(f"DeepChem version: {dc.__version__}")

############################### settings for all experiments #################

best_con = 0.015
best_graph = 0.0143

num_repeats= 3
num_epochs = 2

metric_labels=['mean_squared_error','pearson_r2_score',
               'mae_score', 'rmse']


metric1 = dc.metrics.Metric(dc.metrics.mean_squared_error)
metric2 = dc.metrics.Metric(dc.metrics.pearson_r2_score)
metric3 = dc.metrics.Metric(dc.metrics.mae_score)
metrics = [metric1, metric2, metric3]
selected_metric = 2 #which metric to use for callback

Splitter = dc.splits.RandomSplitter()

# setting up the splitters for the task
#Splitter_Object_tf = dc.splits.SingletaskStratifiedSplitter()
#Splitter_Object_pca = dc.splits.SingletaskStratifiedSplitter()

split_fraction=[0.1, 0.1, 0.8]


TensorFlow version: 2.7.0
DeepChem version: 2.5.0


In [3]:
dataset_name + '_SMILES.csv'

'qm8_SMILES.csv'

In [ ]:
hdf5_file_name=dataset_name + '_topological_features.hdf5'
fh = h5py.File(os.path.join(data_dir,hdf5_file_name), 'r+')
num_of_rows, num_of_molecules = h.basic_info_hdf5_dataset(fh, label='molID')

In [ ]:
fh.keys()

In [ ]:
featurizer_CM_eig = dc.feat.CoulombMatrixEig(max_atoms=23)
tasks, datasets, transformers = loader(
            shard_size=2000,
            featurizer=featurizer_CM_eig,
            splitter=None)

In [ ]:
datasets[0].tasks

In [ ]:
feature_name_list = ['pers_S_1', 'pers_S_2', 'pers_S_3',
                    'no_p_1', 'no_p_2', 'no_p_3',
                    'bottle_1', 'bottle_2', 'bottle_3',
                    'wasser_1', 'wasser_2', 'wasser_3',
                    'landsc_1', 'landsc_2', 'landsc_3',
                    'pers_img_1', 'pers_img_2', 'pers_img_3']

PCA_list = ['PCA_1', 'PCA_2', 'PCA_3',
           'PCA_4', 'PCA_5', 'PCA_6',
           'PCA_7', 'PCA_8', 'PCA_9',
           'PCA_10', 'PCA_11', 'PCA_12',
           'PCA_13', 'PCA_14', 'PCA_15',
           'PCA_16', 'PCA_17', 'PCA_18']

task_list = ['E1-CC2', 'E2-CC2', 'f1-CC2', 'f2-CC2', 
             'E1-PBE0', 'E2-PBE0','f1-PBE0', 'f2-PBE0', 
             'E1-PBE0_2', 'E2-PBE0_2', 'f1-PBE0_2', 'f2-PBE0_2',
             'E1-CAM', 'E2-CAM', 'f1-CAM', 'f2-CAM']

## Functions

In [ ]:
metric_labels

# Load data


## Create topological datasets

We use Numpy datasets to create the two toplogical datasets and two transformers (used as the controls use them and they are supposed to improve training). The topological dataset is normalised in X and y, PCA is only normalised in y as PCA is a normalisation.

This sorts out the datasets, transformers and splitters for the task

In [ ]:
fh['SMILES']

In [ ]:
## loading data from the hdf5 file
X_data = h.load_all_hdf5(
    fh,
    num_of_rows, 
    column_headers=feature_name_list)

PCA_X_data = h.load_all_hdf5(
    fh,
    num_of_rows, 
    column_headers=PCA_list)

y_data = h.load_all_hdf5(
    fh,
    num_of_rows, 
    column_headers=task_list)

SMILES_list = np.array(fh['SMILES'])

# making the datasets
topol_dataset = dc.data.DiskDataset.from_numpy(
    X_data, 
    y_data, 
    ids=SMILES_list)

pca_dataset = dc.data.DiskDataset.from_numpy(
    PCA_X_data, 
    y_data, 
    ids=SMILES_list)

# doing a transform on the data to make it easier for hte NN
# both y and x
transformers_tf = [
    dc.trans.NormalizationTransformer(
        transform_X=True, 
        dataset=topol_dataset),
    dc.trans.NormalizationTransformer(
        transform_y=True, 
        dataset=topol_dataset)]
# only y
transformers_pca = [
    dc.trans.NormalizationTransformer(
        transform_y=True, 
        dataset=pca_dataset)]



In [ ]:
plt.hist(y_data)
plt.xlabel('Energies?')
plt.ylabel("No.")

# Training on the topological features

#### Topological features

In [ ]:
%%time
output_metrics_tf=h.topol_regression_experiment(
    dataset=topol_dataset,
    transformers=transformers_tf,
    Splitter_Object=Splitter,
    tasks=task_list,
    metrics=metrics,
    metric_selector=selected_metric,
    num_repeats=num_repeats,
    num_epochs=num_epochs,
    split_fraction=split_fraction,
    patience=15,
    metric_labels=['mean_squared_error',
                   'pearson_r2_score', 
                   'mae_score', 
                   'rmse']
    )
output_metrics_tf

#### PCA of topological features

In [ ]:
%%time
output_metrics_pca=h.topol_regression_experiment(
    dataset=pca_dataset,
    transformers=transformers_pca,
    Splitter_Object=Splitter,
    tasks=task_list,
    metrics=metrics,
    metric_selector=selected_metric,
    num_repeats=num_repeats,
    num_epochs=num_epochs,
    split_fraction=split_fraction,
    patience=15,
    metric_labels=['mean_squared_error',
                'pearson_r2_score', 
                'mae_score', 
                'rmse']
    )
output_metrics_pca

#### PCA of topological features with no transform

In [ ]:
%%time
output_metrics_pca_no_transform=h.no_transform_topol_regression_experiment(
    dataset=pca_dataset,
    Splitter_Object=Splitter,
    tasks=task_list,
    metrics=metrics,
    metric_selector=selected_metric,
    num_repeats=num_repeats,
    num_epochs=num_epochs,
    split_fraction=split_fraction,
    patience=15,
    metric_labels=['mean_squared_error',
                'pearson_r2_score', 
                'mae_score', 
                'rmse']
    )
output_metrics_pca_no_transform

## 1-D inputs

#### ECFP

In [ ]:
%%time
output_metrics_ecfp = h.deepchem_regression_experiment(
    metrics,
    metric_selector=selected_metric,
    feat_setting='ECFP',
    model_setting='MTR',
    loader=loaders[dataset_name],
    num_repeats=num_repeats,
    num_epochs=num_epochs,
    split_function=Splitter,
    split_fraction=split_fraction,
    patience=15,
    metric_labels=['mean_squared_error',
                   'pearson_r2_score',
                   'mae_score', 
                   'rmse'])
output_metrics_ecfp

### Coulomb matrix eigenvalues

In [ ]:
%%time
output_metrics_cm_eig = h.deepchem_regression_experiment(
    metrics,
    metric_selector=selected_metric,
    feat_setting='CM_eig',
    model_setting='MTR',
    loader=loaders[dataset_name],
    num_repeats=num_repeats,
    num_epochs=num_epochs,
    split_function=Splitter,
    split_fraction=split_fraction,
    patience=15,
    metric_labels=['mean_squared_error',
                   'pearson_r2_score',
                   'mae_score', 
                   'rmse'])

output_metrics_cm_eig

### rdkit features

In [ ]:
%%time
output_metrics_rdkit = h.deepchem_regression_experiment(
    metrics,
    metric_selector=selected_metric,
    feat_setting='rdkit',
    model_setting='MTR',
    loader=loaders[dataset_name],
    num_repeats=num_repeats,
    num_epochs=num_epochs,
    split_function=Splitter,
    split_fraction=split_fraction,
    patience=15,
    metric_labels=['mean_squared_error',
                   'pearson_r2_score',
                   'mae_score', 
                   'rmse'])

output_metrics_rdkit

## MACCS features

In [ ]:
%%time
output_metrics_maccs = h.deepchem_regression_experiment(
    metrics,
    metric_selector=selected_metric,
    feat_setting='MACCS',
    model_setting='MTR',
    loader=loaders[dataset_name],
    num_repeats=num_repeats,
    num_epochs=num_epochs,
    split_function=Splitter,
    split_fraction=split_fraction,
    patience=15,
    metric_labels=['mean_squared_error',
                   'pearson_r2_score',
                   'mae_score', 
                   'rmse'])

output_metrics_maccs

### graph for 1D features

In [ ]:


data=[output_metrics_ecfp['tr_mae'],
     output_metrics_ecfp['te_mae'],
     output_metrics_maccs['tr_mae'],
     output_metrics_maccs['te_mae'],
     output_metrics_rdkit['tr_mae'],
     output_metrics_rdkit['te_mae'],
     output_metrics_cm_eig['tr_mae'],
     output_metrics_cm_eig['te_mae'],
     output_metrics_tf['tr_mae'],
     output_metrics_tf['te_mae'],
     output_metrics_pca['tr_mae'],
     output_metrics_pca['te_mae']]
      
means=[np.mean(x) for x in data]
stds=[np.std(x) for x in data]


x_positions = [x+1 for x in range(len(means))]
plt.figure(figsize=(16, 9))
plt.rcParams.update({'font.size': 22})

plt.bar(x_positions, means, 
        color=["#1f77f4","#f62728",
               "#1f77f4","#f62728",
               "#1f77f4","#f62728",
               "#88fff4","#552200",
               "#5f77f4","#f62788",
               "#5f77f4","#f62788"], 
        edgecolor='k',
        linewidth='3',
        yerr=stds, 
        error_kw=dict(lw=5, capsize=15, capthick=3))
for i in range(len(means)):
    x=data[i]; plt.plot(np.ones(len(x))*(i+1),x,'o',color="#444444")

axes=plt.gca()
#axes.set_ylim([0.5,4.5])
#axes.set_xlim([0.45,4.55])
x_tick_list=[1.5,3.5,5.5,7.5,9.5,11.5]
plt.xticks(x_tick_list,['ECFP',"MACCS","rdkit","CM_eig","TDAF","PCA-TDAF"])
plt.plot([0.5,max(x_tick_list)+1],[best_con, best_con],'k',linewidth=4,linestyle='-.')
#plt.plot([0.5,max(x_tick_list)+1],[best_graph, best_graph],'g',linewidth=4,linestyle='-.')

plt.ylabel('MAE kcal/mol')
plt.savefig(os.path.join(dataset_name + "1Donly.png"))

## 2D inputs and graph models

In [ ]:
%%time
output_metrics_Sm2Img=h.deepchem_regression_experiment(
    metrics,
    metric_selector=selected_metric,
    dimension=2,
    feat_setting='Smiles2Img',
    model_setting='ChemCeption',
    loader=loaders[dataset_name],
    num_repeats=num_repeats,
    num_epochs=num_epochs,
    split_function=Splitter,
    split_fraction=split_fraction,
    patience=15,
    metric_labels=['mean_squared_error',
                   'pearson_r2_score',
                   'mae_score', 
                   'rmse'])

output_metrics_Sm2Img.head(3)

In [ ]:
%%time
output_metrics_gc=h.deepchem_regression_experiment(
    metrics,
    metric_selector=selected_metric,
    dimension=2,
    feat_setting='ConvMol',
    model_setting='GraphConv',
    loader=loaders[dataset_name],
    num_repeats=num_repeats,
    num_epochs=num_epochs,
    split_function=Splitter,
    split_fraction=split_fraction,
    patience=15,
    metric_labels=['mean_squared_error',
                   'pearson_r2_score',
                   'mae_score', 
                   'rmse'])

output_metrics_gc.head(3)

In [ ]:
%%time
output_metrics_weave=h.deepchem_regression_experiment(
    metrics,
    metric_selector=selected_metric,
    dimension=2,
    feat_setting='Weave',
    model_setting='Weave',
    loader=loaders[dataset_name],
    num_repeats=num_repeats,
    num_epochs=num_epochs,
    split_function=Splitter,
    split_fraction=split_fraction,
    patience=15,
    metric_labels=['mean_squared_error',
                   'pearson_r2_score',
                   'mae_score', 
                   'rmse'])

output_metrics_weave.head(3)

In [ ]:
%%time
output_metrics_cm=h.deepchem_regression_experiment(
    metrics,
    metric_selector=selected_metric,
    dimension=2,
    feat_setting='CM',
    model_setting='DTNN',
    loader=loaders[dataset_name],
    num_repeats=num_repeats,
    num_epochs=num_epochs,
    split_function=Splitter,
    split_fraction=split_fraction,
    patience=15,
    metric_labels=['mean_squared_error',
                   'pearson_r2_score',
                   'mae_score', 
                   'rmse'])

output_metrics_cm.head(3)

## Write out data and graphs

In [ ]:
list_of_data_frames=[output_metrics_ecfp,
    output_metrics_maccs,
    output_metrics_rdkit,
    output_metrics_cm_eig,
    output_metrics_Sm2Img,
    output_metrics_gc,
    output_metrics_weave,
    output_metrics_cm,
    output_metrics_tf,
    output_metrics_pca]

list_of_data_frames_names=["output_metrics_ecfp",
    "output_metrics_maccs",
    "output_metrics_rdkit",
    "output_metrics_cm_eig",
    "output_metrics_Sm2Img",
    "output_metrics_gc",
    "output_metrics_weave",
    "output_metrics_cm",
    "output_metrics_tf",
    "output_metrics_pca"]

In [ ]:
for i in range(len(list_of_data_frames)):
    list_of_data_frames[i].to_csv(os.path.join(
        results_dir, 
        list_of_data_frames_names[i] + '.csv'))


In [ ]:
#Best=1.92

data=[output_metrics_ecfp['tr_mae'],
     output_metrics_ecfp['te_mae'],
     output_metrics_maccs['tr_mae'],
     output_metrics_maccs['te_mae'],
     output_metrics_rdkit['tr_mae'],
     output_metrics_rdkit['te_mae'],
     output_metrics_cm_eig['tr_mae'],
     output_metrics_cm_eig['te_mae'],
     output_metrics_Sm2Img['tr_mae'],
     output_metrics_Sm2Img['te_mae'],
     output_metrics_gc['tr_mae'],
     output_metrics_gc['te_mae'],
     output_metrics_weave['tr_mae'],
     output_metrics_weave['te_mae'],
     output_metrics_cm['tr_mae'],
     output_metrics_cm['te_mae']]
     #output_metrics_tf['tr_mae'],
     #output_metrics_tf['te_mae'],
     #output_metrics_pca['tr_mae'],
     #output_metrics_pca['te_mae']]
      
means=[np.mean(x) for x in data]
stds=[np.std(x) for x in data]


x_positions = [x+1 for x in range(len(means))]
plt.figure(figsize=(16, 9))
plt.rcParams.update({'font.size': 22})

plt.bar(x_positions, means, 
        color=["#1f77f4","#f62728", # red blue
               "#1f77f4","#f62728",
               "#1f77f4","#f62728",
               "#88fff4","#552200", #brown green
               "#61ff33","#ffb433", # green orange
               "#61ff33","#ffb433",
               "#61ff33","#ffb433",               
               "#88fff4","#552200"], #brown green               
               #"#5f77f4","#f62788", # pink blue
               #"#5f77f4","#f62788"], 
        edgecolor='k',
        linewidth='3',
        yerr=stds, 
        error_kw=dict(lw=5, capsize=15, capthick=3))
for i in range(len(means)):
    x=data[i]; plt.plot(np.ones(len(x))*(i+1),x,'o',color="#444444")


#axes.set_ylim([0.5,4.5])
#axes.set_xlim([0.45,4.55])
x_tick_list=[1.5,3.5,5.5,7.5,9.5,11.5,13.5,15.5]
plt.xticks(x_tick_list,
           ['ECFP',"MACCS","rdkit","CM_eig",
            'Sm2Img', 'GC', 'Weave', 'CM'], rotation=45)
plt.plot([0.5,max(x_tick_list)+1],[best_con, best_con],'k',linewidth=4,linestyle='-.')
plt.plot([0.5,max(x_tick_list)+1],[best_graph, best_graph],'g',linewidth=4,linestyle='-.')

axes=plt.gca()
plt.ylabel('MAE kcal/mol')
plt.savefig(os.path.join(results_dir, dataset_name + "control_data.png"))

In [ ]:
#Best=1.92

data=[output_metrics_ecfp['tr_mae'],
     output_metrics_ecfp['te_mae'],
     output_metrics_maccs['tr_mae'],
     output_metrics_maccs['te_mae'],
     output_metrics_rdkit['tr_mae'],
     output_metrics_rdkit['te_mae'],
     output_metrics_cm_eig['tr_mae'],
     output_metrics_cm_eig['te_mae'],
     output_metrics_Sm2Img['tr_mae'],
     output_metrics_Sm2Img['te_mae'],
     output_metrics_gc['tr_mae'],
     output_metrics_gc['te_mae'],
     output_metrics_weave['tr_mae'],
     output_metrics_weave['te_mae'],
     output_metrics_cm['tr_mae'],
     output_metrics_cm['te_mae'],
     output_metrics_tf['tr_mae'],
     output_metrics_tf['te_mae'],
     output_metrics_pca['tr_mae'],
     output_metrics_pca['te_mae']]
      
means=[np.mean(x) for x in data]
stds=[np.std(x) for x in data]


x_positions = [x+1 for x in range(len(means))]
plt.figure(figsize=(16, 9))
plt.rcParams.update({'font.size': 22})

plt.bar(x_positions, means, 
        color=["#1f77f4","#f62728", # red blue
               "#1f77f4","#f62728",
               "#1f77f4","#f62728",
               "#88fff4","#552200", #brown green
               "#61ff33","#ffb433", # green orange
               "#61ff33","#ffb433",
               "#61ff33","#ffb433",               
               "#88fff4","#552200", #brown green               
               "#5f77f4","#f62788", # pink blue
               "#5f77f4","#f62788"], 
        edgecolor='k',
        linewidth='3',
        yerr=stds, 
        error_kw=dict(lw=5, capsize=15, capthick=3))
for i in range(len(means)):
    x=data[i]; plt.plot(np.ones(len(x))*(i+1),x,'o',color="#444444")


#axes.set_ylim([0.5,4.5])
#axes.set_xlim([0.45,4.55])
x_tick_list=[1.5,3.5,5.5,7.5,9.5,11.5,13.5,15.5,17.5,19.5]
plt.xticks(x_tick_list,
           ['ECFP',"MACCS","rdkit","CM_eig",
            'Sm2Img', 'GC', 'Weave', 'CM',
            "TDAF","PCA-TDAF"], roation=45)
plt.plot([0.5,max(x_tick_list)+1],[best_con, best_con],'k',linewidth=4,linestyle='-.')
plt.plot([0.5,max(x_tick_list)+1],[best_graph, best_graph],'g',linewidth=4,linestyle='-.')

axes=plt.gca()
plt.ylabel('MAE kcal/mol')
plt.savefig(os.path.join(results_dir, dataset_name + "all_data.png"))

## stats stuff

import scipy
scipy.stats.ttest_ind(test_scores_CM_eig, test_scores_pca, axis=0, equal_var=True, nan_policy='propagate', alternative='two-sided')

In [ ]:
fh.close()